# 04 - Preprocessing and Training

**Objectif :** charger les artefacts transformés par le notebook 03, vérifier le contrat de preprocessing, entraîner les candidats et sélectionner le meilleur modèle.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load processed train and validation partitions

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository
from credit_risk_lab.infrastructure.modeling import CreditRiskPreprocessor

processed_train_df = CSVDatasetRepository(
    CSVDataSourceConfig(path=settings.train_path)
).load()
processed_validation_df = CSVDatasetRepository(
    CSVDataSourceConfig(path=settings.validation_path)
).load()
preprocessor = CreditRiskPreprocessor.load(settings.preprocessing_artifact_path)

print(f"Processed train path: {settings.train_path}")
print(f"Processed validation path: {settings.validation_path}")
print(f"Preprocessor artifact path: {settings.preprocessing_artifact_path}")
print(f"Processed train shape: {processed_train_df.shape}")
print(f"Processed validation shape: {processed_validation_df.shape}")
print(f"Preprocessor output features: {len(preprocessor.feature_names_)}")

{
    "processed_train": processed_train_df.shape,
    "processed_validation": processed_validation_df.shape,
    "preprocessor_features": len(preprocessor.feature_names_),
}

## 2. Build train and validation matrices

In [ ]:
target_column = settings.target_column

x_train = processed_train_df.drop(columns=[target_column])
y_train = processed_train_df[target_column]
x_validation = processed_validation_df.drop(columns=[target_column])
y_validation = processed_validation_df[target_column]

train_schema_matches_preprocessor = x_train.columns.tolist() == preprocessor.feature_names_
validation_schema_matches_train = x_validation.columns.tolist() == x_train.columns.tolist()

if not train_schema_matches_preprocessor:
    raise ValueError("Processed train columns do not match the saved preprocessor features")
if not validation_schema_matches_train:
    raise ValueError("Processed validation columns do not match processed train columns")

print(f"Target column excluded from model features: {target_column}")
print(f"Train feature columns: {x_train.shape[1]}")
print(f"Validation feature columns: {x_validation.shape[1]}")
print(f"Train schema matches saved preprocessor: {train_schema_matches_preprocessor}")
print(f"Validation schema matches train: {validation_schema_matches_train}")
print(f"Final test remains untouched at: {settings.raw_test_path}")

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(y_train), "positive_rate": y_train.mean()},
        {
            "split": "validation",
            "rows": len(y_validation),
            "positive_rate": y_validation.mean(),
        },
    ]
)

display(split_summary)
pd.DataFrame({"model_feature": x_train.columns})

## 3. Inspect configured model candidates

In [ ]:
from credit_risk_lab.infrastructure.modeling import build_configured_models, load_models_config

model_config = load_models_config()
configured_models = build_configured_models(
    random_state=settings.random_state,
    config=model_config,
)

pd.DataFrame(
    [
        {
            "model": model.name,
            "parameters": model.parameters,
            "early_stopping": model.early_stopping_rounds,
        }
        for model in configured_models
    ]
)

## 4. Train candidates

In [ ]:
from credit_risk_lab.infrastructure.modeling import BoostingModelTrainer

trainer = BoostingModelTrainer(random_state=settings.random_state)
training_results = trainer.fit(
    x_train,
    y_train,
    x_validation,
    y_validation,
)

validation_metrics = trainer.results_frame(training_results)
validation_metrics.round(4)

## 5. Select best model

In [ ]:
from credit_risk_lab.infrastructure.modeling import BestModelSelector

selector = BestModelSelector(metric=settings.selection_metric)
best_result = selector.select(training_results)

{
    "selected_model": best_result.model_name,
    "selection_metric": settings.selection_metric,
    "threshold": best_result.threshold,
}

## 6. Training curves

In [ ]:
from credit_risk_lab.infrastructure.visualization import plot_learning_curves, plot_model_comparison

histories = {
    result.model_name: result.history for result in training_results
}

plot_model_comparison(validation_metrics).show()
plot_learning_curves(histories).show()

## 7. Selected model feature importance

In [ ]:
from credit_risk_lab.infrastructure.modeling import CatBoostFeatureImportanceAnalyzer
from credit_risk_lab.infrastructure.visualization import plot_feature_importance

top_n_features = 20

if best_result.model_name == "CatBoost":
    baseline_importance = CatBoostFeatureImportanceAnalyzer(
        best_result.model,
        feature_names=x_train.columns.tolist(),
    ).importance_frame(top_n=top_n_features)
    display(baseline_importance.round(4))
    plot_feature_importance(
        baseline_importance,
        title=f"Top {top_n_features} CatBoost baseline feature importances",
    ).show()
else:
    print(f"Selected baseline model is {best_result.model_name}; CatBoost importance skipped.")